In [1]:
from pathlib import Path
import json
from collections import defaultdict
from typing import Union, Iterable, List
import pandas as pd
from glob import glob

# --------------------------------------------------------
# 0) Load submissions 
# --------------------------------------------------------

# Adjust these paths to your setup
p_sub = Path("sampled.ndjson")

print("Exists sampled submissions:", p_sub.exists(), p_sub.resolve())

# Read NDJSON files into DataFrames
df_submissions = pd.read_json(p_sub, lines=True) if p_sub.exists() else pd.DataFrame()

print("Submissions:", len(df_submissions))

Exists sampled submissions: True /Users/arthur/DataspellProjects/reddit-l/sampled.ndjson
Submissions: 2399


In [2]:
# --------------------------------------------------------
# 1) Comment loader
# --------------------------------------------------------

# Columns to retain from the original Reddit comment JSONL files
COM_KEEP_COLS = [
    "id",
    "author",
    "created_utc",
    "ups",
    "downs",
    "likes",
    "body",
    "parent_id",
    "link_id",
]


def load_comment_jsonl_files(
        paths: Union[str, Path, Iterable[Union[str, Path]]]
) -> pd.DataFrame:
    """
    Load one or multiple Reddit *comment* JSONL files and return a unified DataFrame.

    Functionality:
    - Reads JSON Lines (.jsonl) files
    - Keeps only predefined columns (COM_KEEP_COLS)
    - Fills missing expected columns with NA
    - Converts `created_utc` (Unix timestamp) into a readable datetime format (UTC)
    """

    # If a single path is provided, wrap it into a list
    if isinstance(paths, (str, Path)):
        paths = [paths]

    dfs: List[pd.DataFrame] = []

    for p in paths:
        p = Path(p)

        # Skip non-existing files but continue with the rest
        if not p.exists():
            print(f"Warning: File not found and skipped: {p}")
            continue

        # Read JSON Lines file
        df = pd.read_json(p, lines=True)

        # Ensure all expected columns exist (create missing ones filled with NA)
        missing_cols = [c for c in COM_KEEP_COLS if c not in df.columns]
        for c in missing_cols:
            df[c] = pd.NA

        # Keep only the relevant columns
        df = df[COM_KEEP_COLS].copy()

        # Convert Unix timestamp (seconds since epoch) to UTC datetime
        if "created_utc" in df.columns:
            df["created_utc"] = pd.to_datetime(df["created_utc"], unit="s", utc=True)

        dfs.append(df)

    # If no file was successfully processed, return an empty DataFrame with the correct schema
    if not dfs:
        return pd.DataFrame(columns=COM_KEEP_COLS)

    # Concatenate all loaded DataFrames
    df_all = pd.concat(dfs, ignore_index=True)

    return df_all


# --------------------------------------------------------
# Usage: load all comment files matching data/*_comments.jsonl
# --------------------------------------------------------

comment_files = glob("data/*_comments.jsonl")
print("Found comment files:", len(comment_files))
for f in comment_files:
    print("  -", f)

df_comments = load_comment_jsonl_files(comment_files)

print("Comments   :", len(df_comments))
print(df_comments.head())


Found comment files: 12
  - data/r_worldnews_comments.jsonl
  - data/r_AmItheAsshole_comments.jsonl
  - data/r_TrueReddit_comments.jsonl
  - data/r_PoliticalDiscussion_comments.jsonl
  - data/r_socialjustice_comments.jsonl
  - data/r_politics_comments.jsonl
  - data/r_changemyview_comments.jsonl
  - data/r_AskReddit_comments.jsonl
  - data/r_liberal_comments.jsonl
  - data/r_moderatepolitics_comments.jsonl
  - data/r_debate_comments.jsonl
  - data/r_conservative_comments.jsonl


KeyboardInterrupt: 

In [ ]:
# --------------------------------------------------------
# Build the comment trees with the subkissions that have been sampled 
# Normalize IDs (remove t1_/t3_ prefixes, ensure string type)
# --------------------------------------------------------

def strip_prefix(x):
    """
    Remove the 't1_' / 't3_' etc. prefix from Reddit-style IDs.
    Returns a plain string ID, or None if input is None/NaN.
    """
    if pd.isna(x):
        return None
    x = str(x)
    if "_" in x:
        return x.split("_", 1)[1]
    return x

# Submissions:
# - Many datasets have 'id' already as bare ID (e.g. '1d0ckqi'),
#   but we run strip_prefix defensively.
df_submissions["submission_id"] = df_submissions["id"].apply(strip_prefix)

# Comments:
df_comments["comment_id"] = df_comments["id"].apply(strip_prefix)
df_comments["parent"] = df_comments["parent_id"].apply(strip_prefix)
df_comments["link"] = df_comments["link_id"].apply(strip_prefix)

# --------------------------------------------------------
# 3) Decide which submissions we want (sampled)
# --------------------------------------------------------

# If you already have a sampled DataFrame from your sampler, use that here:
#   df_submissions_sampled = <your sampler output>
#
# For demo purposes we take all submissions. Replace this with your sampled set.
df_submissions_sampled = df_submissions.copy()

sampled_submission_ids = set(df_submissions_sampled["submission_id"])
print("Sampled submissions:", len(sampled_submission_ids))

# --------------------------------------------------------
# 4) Filter comments to only those belonging to sampled submissions
#    (via 'link', which points to the submission ID)
# --------------------------------------------------------

df_comments_in_sample = df_comments[df_comments["link"].isin(sampled_submission_ids)].copy()
print("Comments in sampled submissions:", len(df_comments_in_sample))

# --------------------------------------------------------
# 5) Build parent -> children mapping and comment lookup
# --------------------------------------------------------

# children_map[parent_id] = [child_comment_id, ...]
children_map = defaultdict(list)

for row in df_comments_in_sample.itertuples(index=False):
    # 'parent' is either a submission_id (for top-level comments)
    # or another comment_id (for replies)
    parent_id = row.parent
    comment_id = row.comment_id
    children_map[parent_id].append(comment_id)

# Fast lookup: comment_id -> dict of its fields
comment_lookup = df_comments_in_sample.set_index("comment_id").to_dict(orient="index")

# --------------------------------------------------------
# 6) Recursive function to build a comment tree
# --------------------------------------------------------

def build_comment_tree(comment_id):
    """
    Recursively build a nested comment structure starting from `comment_id`.

    The returned dict looks like:
    {
        "id": ...,
        "parent_id": ...,
        "body": ...,
        "author": ...,
        "score": ...,
        "created_utc": ...,
        "replies": [ ... nested comments ... ]
    }
    """
    c = comment_lookup[comment_id]

    node = {
        "id": comment_id,
        "parent_id": c.get("parent"),
        "body": c.get("body"),
        "author": c.get("author"),
        "score": c.get("score"),
        "created_utc": c.get("created_utc"),
        "permalink": c.get("permalink"),
        "subreddit": c.get("subreddit"),
        # Add any additional fields you need here
        "replies": []
    }

    # Recursively build all children
    for child_id in children_map.get(comment_id, []):
        node["replies"].append(build_comment_tree(child_id))

    return node

# --------------------------------------------------------
# 7) Build thread structure per submission
# --------------------------------------------------------

# Fast lookup for submission metadata by submission_id
submissions_lookup = df_submissions_sampled.set_index("submission_id").to_dict(orient="index")

threads = []

for sid, srow in submissions_lookup.items():
    # Top-level comments: parent == submission_id
    top_level_comment_ids = children_map.get(sid, [])

    thread = {
        "submission_id": sid,
        "title": srow.get("title"),
        "selftext": srow.get("selftext"),
        "author": srow.get("author"),
        "score": srow.get("score"),
        "created_utc": srow.get("created_utc"),
        "subreddit": srow.get("subreddit"),
        "url": srow.get("url"),
        # Nested comment tree
        "comments": [build_comment_tree(cid) for cid in top_level_comment_ids],
    }

    threads.append(thread)

print("Built threads:", len(threads))

# Optional: if you prefer a dict keyed by submission_id instead of a list:
# threads_by_submission = {t["submission_id"]: t for t in threads}

# --------------------------------------------------------
# 8) Save threads to JSON file
# --------------------------------------------------------

out_path = Path("conversation_threads.json")

with out_path.open("w", encoding="utf-8") as f:
    json.dump(threads, f, ensure_ascii=False, indent=2)

print("Threads written to:", out_path.resolve())
